In [2]:
from ultralytics import YOLO
import cv2

model = YOLO(r"D:\New folder\best.pt")
VEHICLE_CLASSES = set(model.names.values())

def process_video(video_path, output_path="output.mp4", line_y=200):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video file at {video_path}")

    # Process at 720p for speed
    target_w, target_h = 1280, 720
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (target_w, target_h))

    counts = {cls: 0 for cls in VEHICLE_CLASSES}
    counted_ids = set()

    # Pass imgsz=640 to model.track to force fast inference resolution
    results = model.track(source=video_path, imgsz=640, persist=True, stream=True, verbose=False)

    for r in results:
        # Resize frame immediately to cut processing time
        frame = cv2.resize(r.orig_img, (target_w, target_h))
        
        cv2.line(frame, (0, line_y), (target_w, line_y), (0, 255, 255), 2)

        if r.boxes is not None and r.boxes.id is not None:
            # Calculate scaling ratios to map original box coordinates to 720p
            orig_h, orig_w = r.orig_img.shape[:2]
            scale_x = target_w / orig_w
            scale_y = target_h / orig_h

            for box, track_id, cls_id in zip(r.boxes.xyxy, r.boxes.id, r.boxes.cls):
                x1, y1, x2, y2 = box.tolist()
                
                # Scale coordinates
                x1, x2 = int(x1 * scale_x), int(x2 * scale_x)
                y1, y2 = int(y1 * scale_y), int(y2 * scale_y)
                
                cls_name = model.names[int(cls_id)]
                uid = int(track_id)
                cy = (y1 + y2) // 2

                if uid not in counted_ids and cy > line_y:
                    counted_ids.add(uid)
                    counts[cls_name] += 1

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
                cv2.putText(frame, f"{cls_name} #{uid}", (x1, max(y1 - 8, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 2)

        # Draw overlays
        y_offset = 30
        for cls, n in counts.items():
            cv2.putText(frame, f"{cls}: {n}", (10, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            y_offset += 25

        cv2.imshow("Traffic Monitoring", frame)
        out.write(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    return output_path, counts

output_path, final_counts = process_video(
    r"D:\New folder\18437773-uhd_3840_2160_50fps.mp4", 
    line_y=200
)
print("\nFinal Vehicle Counts:", final_counts)


Final Vehicle Counts: {'Bus': 2, 'truck': 3, 'Motorcycle': 0, 'car': 29, '0': 0}


In [5]:
import time
cap = cv2.VideoCapture(r"D:\New folder\18437773-uhd_3840_2160_50fps.mp4")
n_frames = 0
start = time.time()
for r in model.track(source=r"D:\New folder\18437773-uhd_3840_2160_50fps.mp4", stream=True, verbose=False):
    n_frames += 1
elapsed = time.time() - start
print(f"FPS: {n_frames / elapsed:.2f}")

FPS: 2.75


In [8]:
metrics = model.val(data=r"D:\New folder\data.yaml")
print(metrics.box.map, metrics.box.map50, metrics.box.mp, metrics.box.mr)

Ultralytics 8.4.108  Python-3.11.9 torch-2.13.0+cpu CPU (Intel Core i5-1035G1 1.00GHz)
val: Fast image access  (ping: 0.10.1 ms, read: 121.358.8 MB/s, size: 55.1 KB)
val: Scanning D:\New folder\valid\labels... 220 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 220/220 1.3Kit/s 0.2s<0.3s
val: New cache created: D:\New folder\valid\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.7s/it 37.3s3.0ss
                   all        220        293      0.845      0.795      0.823      0.635
                   Bus         24         32      0.928      0.906      0.904      0.826
            Motorcycle          6          7      0.656      0.548      0.589      0.244
                   car        150        203      0.868      0.823      0.867      0.667
                 truck         50         51       0.93      0.902      0.932      0.805
Speed: 3.6ms preprocess, 150.9ms inference, 0.0ms loss, 1.0ms postproces

In [1]:
import gradio as gr
import cv2
import os
import numpy as np
from ultralytics import YOLO

# Load model
model = YOLO(r"D:\New folder\best.pt")
VEHICLE_CLASSES = set(model.names.values())

def stream_video_gradio(video_path, line_y=200):
    if not video_path or not os.path.exists(video_path):
        yield None, "Error: Invalid video file."
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        yield None, "Error: Could not open video stream."
        return

    target_w, target_h = 1280, 720
    counts = {cls: 0 for cls in VEHICLE_CLASSES}
    counted_ids = set()

    results = model.track(
        source=video_path, 
        imgsz=640, 
        conf=0.15, 
        persist=True, 
        stream=True, 
        verbose=False
    )

    for r in results:
        frame = cv2.resize(r.orig_img, (target_w, target_h))
        cv2.line(frame, (0, line_y), (target_w, line_y), (0, 255, 255), 2)

        if r.boxes is not None and r.boxes.id is not None:
            orig_h, orig_w = r.orig_img.shape[:2]
            scale_x, scale_y = target_w / orig_w, target_h / orig_h

            for box, track_id, cls_id in zip(r.boxes.xyxy, r.boxes.id, r.boxes.cls):
                x1, y1, x2, y2 = box.tolist()
                x1, x2 = int(x1 * scale_x), int(x2 * scale_x)
                y1, y2 = int(y1 * scale_y), int(y2 * scale_y)
                
                cls_name = model.names[int(cls_id)]
                uid = int(track_id)
                cy = (y1 + y2) // 2

                if uid not in counted_ids and cy > line_y:
                    counted_ids.add(uid)
                    counts[cls_name] += 1

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
                cv2.putText(frame, f"{cls_name} #{uid}", (x1, max(y1 - 8, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 2)

        # Overlay totals
        y_offset = 30
        for cls, n in counts.items():
            cv2.putText(frame, f"{cls}: {n}", (10, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            y_offset += 25

        # Convert OpenCV BGR image format to RGB for Gradio Image component
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        summary = "--- LIVE VEHICLE COUNT ---\n" + "\n".join(f"{k.capitalize()}: {v}" for k, v in counts.items())

        # Yield frame frame-by-frame live into the web GUI!
        yield frame_rgb, summary

    cap.release()

# Interface setup using Image output for real-time frame streaming
demo = gr.Interface(
    fn=stream_video_gradio,
    inputs=gr.Video(label="Upload Traffic Video"),
    outputs=[
        gr.Image(label="Real-Time Detection Feed"),
        gr.Textbox(label="Live Count Tally")
    ],
    title="Real-Time Traffic Analytics",
    description="Upload a video to see live bounding boxes, object tracking, and line counts updating frame-by-frame."
)

if __name__ == "__main__":
    demo.launch()

c:\PythonEnvs\MasterEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
